### Model Training

In [405]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
# model selection
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
# preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
# models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import GradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, recall_score, classification_report,roc_auc_score, precision_score, f1_score, confusion_matrix


### 1. Load the Dataset

In [406]:
df = pd.read_csv("data/clean_breast_cancer.csv")
df.head()

,Age,Race,Marital Status,T Stage,N Stage,6th Stage,Tumor Size,Estrogen Status,Progesterone Status,Regional Node Examined,Reginol Node Positive,Status
0,68,White,Married,T1,N1,IIA,4,Positive,Positive,24,1,Alive
1,50,White,Married,T2,N2,IIIA,35,Positive,Positive,14,5,Alive
2,58,White,Divorced,T3,N3,IIIC,63,Positive,Positive,14,7,Alive
3,58,White,Married,T1,N1,IIA,18,Positive,Positive,2,1,Alive
4,47,White,Married,T2,N1,IIB,41,Positive,Positive,3,1,Alive


### 2. Encoding

In [407]:
# Label Encoding
df["Status"] = df["Status"].map({"Alive":1 , "Dead":0})
df["Race"] = df["Race"].map({"White":1 ,"Black":2 ,"Other":3})

df["Progesterone Status"] = df["Progesterone Status"].map({"Positive":1 ,"Negative":0})

df["Estrogen Status"] = df["Estrogen Status"].map({"Positive":1 ,"Negative":0})

# One-Hot Encoding

cat_cols = [
    "Marital Status",
    "6th Stage",
    "T Stage",
    "N Stage"
]

df = pd.get_dummies(df, columns=cat_cols, drop_first=True)


### 3. test and train data

In [408]:
x = df.drop("Status", axis=1)
y = df["Status"]

In [409]:
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.8, random_state=42, stratify=y)

### 4. StandardScaler

In [410]:
df

,Age,Race,Tumor Size,Estrogen Status,Progesterone Status,Regional Node Examined,Reginol Node Positive,Status,Marital Status_Married,Marital Status_Separated,...,Marital Status_Widowed,6th Stage_IIB,6th Stage_IIIA,6th Stage_IIIB,6th Stage_IIIC,T Stage_T2,T Stage_T3,T Stage_T4,N Stage_N2,N Stage_N3
0,68,1,4,1,1,24,1,1,True,False,...,False,False,False,False,False,False,False,False,False,False
1,50,1,35,1,1,14,5,1,True,False,...,False,False,True,False,False,True,False,False,True,False
2,58,1,63,1,1,14,7,1,False,False,...,False,False,False,False,True,False,True,False,False,True
3,58,1,18,1,1,2,1,1,True,False,...,False,False,False,False,False,False,False,False,False,False
4,47,1,41,1,1,3,1,1,True,False,...,False,True,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4018,62,3,9,1,1,1,1,1,True,False,...,False,False,False,False,False,False,False,False,False,False
4019,56,1,46,1,1,14,8,1,False,False,...,False,False,True,False,False,True,False,False,True,False
4020,68,1,22,1,0,11,3,1,True,False,...,False,True,False,False,False,True,False,False,False,False
4021,58,2,44,1,1,11,1,1,False,False,...,False,True,False,False,False,True,False,False,False,False


In [411]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)


In [412]:
df

,Age,Race,Tumor Size,Estrogen Status,Progesterone Status,Regional Node Examined,Reginol Node Positive,Status,Marital Status_Married,Marital Status_Separated,...,Marital Status_Widowed,6th Stage_IIB,6th Stage_IIIA,6th Stage_IIIB,6th Stage_IIIC,T Stage_T2,T Stage_T3,T Stage_T4,N Stage_N2,N Stage_N3
0,68,1,4,1,1,24,1,1,True,False,...,False,False,False,False,False,False,False,False,False,False
1,50,1,35,1,1,14,5,1,True,False,...,False,False,True,False,False,True,False,False,True,False
2,58,1,63,1,1,14,7,1,False,False,...,False,False,False,False,True,False,True,False,False,True
3,58,1,18,1,1,2,1,1,True,False,...,False,False,False,False,False,False,False,False,False,False
4,47,1,41,1,1,3,1,1,True,False,...,False,True,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4018,62,3,9,1,1,1,1,1,True,False,...,False,False,False,False,False,False,False,False,False,False
4019,56,1,46,1,1,14,8,1,False,False,...,False,False,True,False,False,True,False,False,True,False
4020,68,1,22,1,0,11,3,1,True,False,...,False,True,False,False,False,True,False,False,False,False
4021,58,2,44,1,1,11,1,1,False,False,...,False,True,False,False,False,True,False,False,False,False


### 5. SMOTE

In [413]:
smote = SMOTE(random_state=42)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train_scaled, y_train)

### MODEL TRAINING

### 1. LogisticRegression

In [414]:
# #logistic
# model = LogisticRegression()
# paramters = {
#     "C": [0.01, 0.1, 1, 10, 100],
#     "penalty": ["l1", "l2"],
#     "solver": ["liblinear"]
# }
# grid_search = GridSearchCV(estimator=model, param_grid=paramters, cv=5, n_jobs=-1, verbose=2)
# grid_search.fit(x_train_resampled, y_train_resampled)   
# print("Best Parameters:", grid_search.best_params_)
# print("Best Cross-Validation Score:", grid_search.best_score_)

In [415]:
model = LogisticRegression(
    class_weight="balanced",
    C=0.1,
    penalty="l2",
    solver="liblinear"
)
model.fit(x_train_resampled, y_train_resampled)

LogisticRegression(C=0.1, class_weight='balanced', solver='liblinear')

In [416]:
y_prob = model.predict_proba(x_test_scaled)[:, 1]
y_test_pred = (y_prob >= 0.4).astype(int)

In [417]:
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("Report:", classification_report(y_test, y_test_pred))
print("precision:", precision_score(y_test, y_test_pred))
print("recall:", recall_score(y_test, y_test_pred))
print("f1_score:", f1_score(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))


Test Accuracy: 0.782608695652174
Report:               precision    recall  f1-score   support

           0       0.33      0.41      0.37       123
           1       0.89      0.85      0.87       682

    accuracy                           0.78       805
   macro avg       0.61      0.63      0.62       805
weighted avg       0.80      0.78      0.79       805

precision: 0.8894009216589862
recall: 0.8489736070381232
f1_score: 0.8687171792948237
[[ 51  72]
 [103 579]]


### 2. RandomForest

In [418]:
# model1 = RandomForestClassifier()
# paramters = {
#     "n_estimators": [100, 200, 300, 400],
#     "max_depth": [None, 10, 20, 30],
#     "min_samples_split": [2, 5, 10],
#     "min_samples_leaf": [1, 2, 4]
# }
# grid_search_rf = GridSearchCV(estimator=model1, param_grid=paramters, cv=3, n_jobs=-1, verbose=2)
# grid_search_rf.fit(x_train_resampled, y_train_resampled)
# print("Best Parameters:", grid_search_rf.best_params_)
# print("Best Cross-Validation Score:", grid_search_rf.best_score_)

In [419]:
model1 = RandomForestClassifier(
    class_weight="balanced",
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1
)
model1.fit(x_train_resampled, y_train_resampled)


RandomForestClassifier(class_weight='balanced', n_estimators=300)

In [420]:
y_prob1 = model1.predict_proba(x_test_scaled)[:, 1]
y_test_pred1 = (y_prob1 >= 0.4).astype(int)

In [421]:
print("Test Accuracy:", accuracy_score(y_test, y_test_pred1))
print("Report:", classification_report(y_test, y_test_pred1))
print("precision:", precision_score(y_test, y_test_pred1))
print("recall:", recall_score(y_test, y_test_pred1))
print("f1_score:", f1_score(y_test, y_test_pred1))
print(confusion_matrix(y_test, y_test_pred1))

Test Accuracy: 0.8335403726708075
Report:               precision    recall  f1-score   support

           0       0.42      0.23      0.29       123
           1       0.87      0.94      0.91       682

    accuracy                           0.83       805
   macro avg       0.64      0.59      0.60       805
weighted avg       0.80      0.83      0.81       805

precision: 0.8712737127371274
recall: 0.9428152492668622
f1_score: 0.9056338028169014
[[ 28  95]
 [ 39 643]]


### 3. XGBoost Classifier

In [422]:
# model2 = XGBClassifier()
# paramters = {
#     'n_estimators': [100, 200],
#     'max_depth': [3, 5, 7],
#     'learning_rate': [0.01, 0.1, 0.2],
#     'subsample': [0.8, 1.0],
#     'colsample_bytree': [0.8, 1.0]
# }
# grid_search_xgb = GridSearchCV(estimator=model2, param_grid=paramters, cv=3, n_jobs=-1, verbose=1)
# grid_search_xgb.fit(x_train_resampled, y_train_resampled)

In [423]:
# grid_search_xgb.best_params_

In [424]:
model2 = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.2,
    subsample=1.0,
    colsample_bytree=0.8
)
model2.fit(x_train_resampled, y_train_resampled)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.2, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [425]:
y_prob = model2.predict_proba(x_test_scaled)[:, 1]
y_test_pred2 = (y_prob >= 0.4).astype(int)

In [426]:
print("Test Accuracy:", accuracy_score(y_test, y_test_pred2))
print("Report:", classification_report(y_test, y_test_pred2))
print("precision:", precision_score(y_test, y_test_pred2))
print("recall:", recall_score(y_test, y_test_pred2))
print("f1_score:", f1_score(y_test, y_test_pred2))
print(confusion_matrix(y_test, y_test_pred2))

Test Accuracy: 0.8422360248447205
Report:               precision    recall  f1-score   support

           0       0.45      0.14      0.21       123
           1       0.86      0.97      0.91       682

    accuracy                           0.84       805
   macro avg       0.65      0.55      0.56       805
weighted avg       0.80      0.84      0.81       805

precision: 0.8617992177314211
recall: 0.969208211143695
f1_score: 0.9123533471359558
[[ 17 106]
 [ 21 661]]


### 4. DecisionTree

In [427]:
model3 = DecisionTreeClassifier(
    class_weight="balanced", 
    max_depth=5, 
    random_state=42,
    min_samples_leaf=15,

)
model3.fit(x_train_resampled, y_train_resampled) 

DecisionTreeClassifier(class_weight='balanced', max_depth=5,
                       min_samples_leaf=15, random_state=42)

In [428]:
y_prob2 = model3.predict_proba(x_test_scaled)[:, 1]
y_test_pred3 = (y_prob2 >= 0.4).astype(int)

In [429]:
print("Test Accuracy:", accuracy_score(y_test, y_test_pred3))
print("Report:", classification_report(y_test, y_test_pred3))
print("precision:", precision_score(y_test, y_test_pred3))
print("recall:", recall_score(y_test, y_test_pred3))
print("f1_score:", f1_score(y_test, y_test_pred3))
print(confusion_matrix(y_test, y_test_pred3))

Test Accuracy: 0.7776397515527951
Report:               precision    recall  f1-score   support

           0       0.34      0.50      0.41       123
           1       0.90      0.83      0.86       682

    accuracy                           0.78       805
   macro avg       0.62      0.66      0.63       805
weighted avg       0.82      0.78      0.79       805

precision: 0.9011164274322169
recall: 0.8284457478005866
f1_score: 0.8632543926661573
[[ 61  62]
 [117 565]]


### 5. SVM model

In [430]:
model4 = SVC(class_weight="balanced", probability=True, random_state=42)
model4.fit(x_train_resampled, y_train_resampled)

SVC(class_weight='balanced', probability=True, random_state=42)

In [431]:
y_prob = model4.predict_proba(x_test_scaled)[:, 1]
y_test_pred4 = (y_prob >= 0.39).astype(int)

In [432]:
print("Test Accuracy:", accuracy_score(y_test, y_test_pred4))
print("Report:", classification_report(y_test, y_test_pred4))
print("precision:", precision_score(y_test, y_test_pred4))
print("recall:", recall_score(y_test, y_test_pred4))
print("f1_score:", f1_score(y_test, y_test_pred4))
print(confusion_matrix(y_test, y_test_pred4))

Test Accuracy: 0.7664596273291926
Report:               precision    recall  f1-score   support

           0       0.32      0.49      0.39       123
           1       0.90      0.82      0.86       682

    accuracy                           0.77       805
   macro avg       0.61      0.65      0.62       805
weighted avg       0.81      0.77      0.78       805

precision: 0.8983870967741936
recall: 0.8167155425219942
f1_score: 0.8556067588325653
[[ 60  63]
 [125 557]]


### 6. GradientBoosting

In [433]:
# model5 = GradientBoostingClassifier(random_state=42)
# paramters = {
#     'loss': [ 'log_loss', 'exponential'],
#     'learning_rate': [0.001, 0.1, 1, 10],
#     'n_estimators': [100, 150, 180, 200]
# }
# grid_search = GridSearchCV(model5, paramters, cv=20, n_jobs=-1, verbose=1)
# grid_search.fit(x_train_resampled, y_train_resampled)
# print("Best Parameters:", grid_search.best_params_)

In [434]:
model5 = GradientBoostingClassifier(
    random_state=42,
    loss='exponential',
    learning_rate=0.5,
    n_estimators=100
)
model5.fit(x_train_resampled, y_train_resampled)

GradientBoostingClassifier(learning_rate=0.5, loss='exponential',
                           random_state=42)

In [435]:
y_prob2 = model5.predict_proba(x_test_scaled)[:, 1]
y_test_pred5 = (y_prob2 >= 0.4).astype(int)

In [436]:
print("Test Accuracy:", accuracy_score(y_test, y_test_pred5))
print("Report:", classification_report(y_test, y_test_pred5))
print("precision:", precision_score(y_test, y_test_pred5))
print("recall:", recall_score(y_test, y_test_pred5))
print("f1_score:", f1_score(y_test, y_test_pred5))
print(confusion_matrix(y_test, y_test_pred5))

Test Accuracy: 0.8397515527950311
Report:               precision    recall  f1-score   support

           0       0.42      0.14      0.21       123
           1       0.86      0.97      0.91       682

    accuracy                           0.84       805
   macro avg       0.64      0.55      0.56       805
weighted avg       0.79      0.84      0.80       805

precision: 0.8614379084967321
recall: 0.966275659824047
f1_score: 0.9108500345542502
[[ 17 106]
 [ 23 659]]


### 7. CatBoost Model

In [437]:
# grid_search_catboost.best_params_

In [438]:
model6 = CatBoostClassifier(
    random_state=42,
    depth=7,
    iterations=200,
    learning_rate=0.1,
    verbose=0
)
model6.fit(x_train_resampled, y_train_resampled)

CatBoostClassifier(depth=7, iterations=200, learning_rate=0.1, random_state=42, verbose=0)

In [439]:
y_prob = model6.predict_proba(x_test_scaled)[:, 1]
y_test_pred6 = (y_prob >= 0.4).astype(int)

In [440]:
print("Test Accuracy:", accuracy_score(y_test, y_test_pred6))
print("Report:", classification_report(y_test, y_test_pred6))
print("precision:", precision_score(y_test, y_test_pred6))
print("recall:", recall_score(y_test, y_test_pred6))
print("f1_score:", f1_score(y_test, y_test_pred6))
print(confusion_matrix(y_test, y_test_pred6))

Test Accuracy: 0.8422360248447205
Report:               precision    recall  f1-score   support

           0       0.45      0.14      0.21       123
           1       0.86      0.97      0.91       682

    accuracy                           0.84       805
   macro avg       0.65      0.55      0.56       805
weighted avg       0.80      0.84      0.81       805

precision: 0.8617992177314211
recall: 0.969208211143695
f1_score: 0.9123533471359558
[[ 17 106]
 [ 21 661]]


### Chosen BEST Model 

In [441]:
# Evaluating AUC for the models
#1. Logistic Regression
print("1. Logistic Regression")
y_prob1 = model.predict_proba(x_test_scaled)[:,1]
auc = roc_auc_score(y_test, y_prob1)
print("AUC:", auc)

print("Train Score:", model.score(x_train_resampled, y_train_resampled))
print("Test Score:", model.score(x_test_scaled, y_test))

print("===============================")

#2. DECISION TREEE
print("2. DECISION TREE")
y_prob2 = model3.predict_proba(x_test_scaled)[:,1]
auc = roc_auc_score(y_test, y_prob1)
print("AUC:", auc)

print("Train Score:", model3.score(x_train_resampled, y_train_resampled))
print("Test Score:", model3.score(x_test_scaled, y_test))

1. Logistic Regression
AUC: 0.7466084924778866
Train Score: 0.6722935779816513
Test Score: 0.7316770186335404
2. DECISION TREE
AUC: 0.7466084924778866
Train Score: 0.7337614678899083
Test Score: 0.7080745341614907


In [442]:

importance = pd.DataFrame({
    "Feature": x.columns,
    "Coefficient": model.coef_[0]
})

importance.sort_values(
    by="Coefficient",
    ascending=False
).head(10)

,Feature,Coefficient
5,Regional Node Examined,0.275221
4,Progesterone Status,0.260494
3,Estrogen Status,0.223820
7,Marital Status_Married,0.041857
1,Race,0.021685
15,T Stage_T2,0.017365
10,Marital Status_Widowed,-0.015396
12,6th Stage_IIIA,-0.024028
13,6th Stage_IIIB,-0.024436
16,T Stage_T3,-0.053040


In [443]:
importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": model3.feature_importances_
})

importance.sort_values(
    by="Importance",
    ascending=False
).head(10)

,Feature,Importance
6,Reginol Node Positive,0.689251
4,Progesterone Status,0.174044
1,Race,0.035022
2,Tumor Size,0.032097
7,Marital Status_Married,0.030308
3,Estrogen Status,0.027241
0,Age,0.012037
13,6th Stage_IIIB,0.000000
18,N Stage_N2,0.000000
17,T Stage_T4,0.000000


In [444]:
import matplotlib.pyplot as plt

top10.plot(
    x="Feature",
    y="Importance",
    kind="barh",
    figsize=(7,5)
)

plt.title("Top 10 Important Features")

plt.show()

NameError: name 'top10' is not defined

### Predction on Sample Input

In [445]:
sample_input1 = [{
    "Age": 40,
    "Race": "White",
    "Marital Status": "Married",
    "T Stage": "T2",
    "N Stage": "N1",
    "6th Stage": "IIB",
    "Tumor Size": 30,
    "Estrogen Status": "Positive",
    "Progesterone Status": "Positive",
    "Regional Node Examined": 9,
    "Reginol Node Positive": 1
}]
sample_input_df = pd.DataFrame(sample_input1, columns=x.columns)
sample_input_scaled = scaler.transform(sample_input_df)
prediction = model.predict(sample_input_scaled)
prob = model.predict_proba(sample_input_scaled)

ValueError: could not convert string to float: 'White'

In [ ]:
sample_input_df = pd.DataFrame(sample_input, columns=x.columns)
sample_input_scaled = scaler.transform(sample_input_df)
prediction = model.predict(sample_input_scaled)
prob = model.predict_proba(sample_input_scaled)

In [ ]:
print(prediction)  # 1= Alive, 0= Dead 
print(prob) # Probability of being in each class (Alive, Dead)

[0]
[[0.97949221 0.02050779]]


In [463]:
sample_input = [[
    70,  # Age (high)
    1,   # Race
    80,  # Tumor Size (very high)
    0,   # Estrogen Status (negative)
    0,   # Progesterone Status (negative)
    25,  # Regional Node Examined (high spread)
    5,   # Reginol Node Positive (very high)

    # Marital Status
    0,   # Married
    0,   # Separated
    0,   # Single
    1,   # Widowed

    # 6th Stage (advanced stage)
    0,   # IIB
    1,   # IIIA
    1,   # IIIB
    1,   # IIIC

    # T Stage (advanced tumor)
    0,   # T2
    0,   # T3
    1,   # T4

    # N Stage (advanced node involvement)
    1,   # N2
    1    # N3
]]
sample_input_df = pd.DataFrame(sample_input, columns=x.columns)
sample_input_scaled = scaler.transform(sample_input_df)
prediction = model3.predict(sample_input_scaled)
prob = model3.predict_proba(sample_input_scaled)

print(prediction)  # 1= Alive, 0= Dead 

dead_prob = prob[0][0]
alive_prob = prob[0][1]

print("Dead Probability:", dead_prob)
print("Alive Probability:", alive_prob)

[0]
Dead Probability: 0.9037037037037037
Alive Probability: 0.0962962962962963


In [450]:
print(x.columns)
print(len(x.columns))


Index(['Age', 'Race', 'Tumor Size', 'Estrogen Status', 'Progesterone Status',
       'Regional Node Examined', 'Reginol Node Positive',
       'Marital Status_Married', 'Marital Status_Separated',
       'Marital Status_Single ', 'Marital Status_Widowed', '6th Stage_IIB',
       '6th Stage_IIIA', '6th Stage_IIIB', '6th Stage_IIIC', 'T Stage_T2',
       'T Stage_T3', 'T Stage_T4', 'N Stage_N2', 'N Stage_N3'],
      dtype='object')
20
